# Hybrid Diffusion-Based X-ray Image Enhancement

**Pipeline:** Anisotropic Diffusion → DnCNN Deep Denoiser → CLAHE → Optional Sharpening → Quality Evaluation

This notebook is the **Colab-compatible version** of the desktop application. It writes the exact same
modular `.py` files used by the desktop app to disk (via `%%writefile`), then imports and runs them here —
so there is a single source of truth for the algorithms, and no drift between the two versions.

**How to use:**
1. Run all setup cells (Sections 1–2) top to bottom once.
2. Run the **Upload** cell and choose an X-ray image (PNG/JPG/JPEG/BMP/TIFF).
3. Run the **Process** cell.
4. View results, metrics, and download the enhanced image.

> Runs entirely on CPU by default; if you enable a GPU runtime (Runtime → Change runtime type → GPU),
> the DnCNN stage will automatically use it.


## 1. Setup: install dependencies and create the project folder


In [ ]:
!pip install -q scikit-image opencv-python-headless PyWavelets

import os
PROJECT_DIR = "/content/xray_enhancer"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/models", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/outputs", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/sample_images", exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())


## 2. Write the pipeline modules to disk
Each cell below is identical to the corresponding file in the desktop project (`config.py`, `utils.py`, ... `pipeline.py`). Run them in order once per session.


In [ ]:
%%writefile config.py
"""
config.py
=========
Central configuration for the Hybrid X-ray Image Enhancement pipeline.

Keeping every tunable parameter in one place makes the pipeline easy to
tune from the GUI, from a script, or from a future config file (JSON/YAML)
without touching the algorithm code itself.
"""

from dataclasses import dataclass, field
from pathlib import Path


# --------------------------------------------------------------------------
# Project paths
# --------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SAMPLE_IMAGES_DIR = PROJECT_ROOT / "sample_images"
LOG_DIR = PROJECT_ROOT / "logs"

for _dir in (MODELS_DIR, OUTPUTS_DIR, SAMPLE_IMAGES_DIR, LOG_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# Path where a pretrained DnCNN checkpoint is expected. If it is not found,
# the deep-learning stage is skipped gracefully (see deep_denoiser.py).
DNCNN_WEIGHTS_PATH = MODELS_DIR / "dncnn.pth"

SUPPORTED_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")


@dataclass
class AnisotropicDiffusionConfig:
    """Parameters for the Perona-Malik anisotropic diffusion stage."""
    num_iterations: int = 15
    kappa: float = 30.0          # conductance / edge-sensitivity threshold
    lam: float = 0.20            # diffusion rate, must be <= 0.25 for 2D stability
    option: int = 1              # 1 = exponential conductance, 2 = quadratic conductance


@dataclass
class DnCNNConfig:
    """Parameters for the deep-learning denoising stage."""
    depth: int = 17               # number of conv layers (DnCNN-S default)
    num_channels: int = 64
    use_gpu_if_available: bool = True
    weights_path: Path = DNCNN_WEIGHTS_PATH


@dataclass
class CLAHEConfig:
    """Parameters for Contrast Limited Adaptive Histogram Equalization."""
    clip_limit: float = 2.5
    tile_grid_size: tuple = (8, 8)


@dataclass
class SharpeningConfig:
    """Parameters for the optional edge-enhancement stage."""
    enabled: bool = True
    method: str = "unsharp_mask"   # "unsharp_mask" or "laplacian"
    amount: float = 0.6            # strength of unsharp mask
    radius: float = 2.0            # Gaussian blur radius used to build the mask
    edge_gain_guard: float = 1.15  # if sharpening raises noise-metric beyond this
                                    # factor, the stage is auto-skipped (see enhancement.py)


@dataclass
class PipelineConfig:
    """Top-level configuration bundling every stage's parameters."""
    resize_max_dim: int = 1024     # safety cap so very large images stay fast; 0 disables
    anisotropic: AnisotropicDiffusionConfig = field(default_factory=AnisotropicDiffusionConfig)
    dncnn: DnCNNConfig = field(default_factory=DnCNNConfig)
    clahe: CLAHEConfig = field(default_factory=CLAHEConfig)
    sharpen: SharpeningConfig = field(default_factory=SharpeningConfig)


DEFAULT_CONFIG = PipelineConfig()


In [ ]:
%%writefile utils.py
"""
utils.py
========
Shared helpers: logging setup and small numeric utilities reused across
several pipeline stages. Centralising them avoids duplicated code and
keeps each stage module focused on its own algorithm.
"""

from __future__ import annotations

import logging
import sys
import time
from contextlib import contextmanager
from typing import Iterator

import numpy as np

from config import LOG_DIR


def get_logger(name: str) -> logging.Logger:
    """
    Create (or fetch) a module-level logger that writes to both the
    console and a rotating log file under logs/.

    Using a named logger per module (via __name__) makes it possible to
    trace exactly which pipeline stage produced a given message.
    """
    logger = logging.getLogger(name)
    if logger.handlers:
        # Avoid adding duplicate handlers if this is called more than once
        # (e.g. GUI re-imports modules on hot reload).
        return logger

    logger.setLevel(logging.DEBUG)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(LOG_DIR / "pipeline.log", encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger


@contextmanager
def timer(label: str, logger: logging.Logger | None = None) -> Iterator[dict]:
    """
    Context manager that measures wall-clock time for a block of code.

    Usage:
        with timer("CLAHE stage", logger) as t:
            ... do work ...
        print(t["elapsed"])

    Returns a dict (mutated in place) so the elapsed time is available
    after the `with` block exits, which is convenient for the GUI's
    per-stage timing display.
    """
    result = {"elapsed": None}
    start = time.perf_counter()
    try:
        yield result
    finally:
        elapsed = time.perf_counter() - start
        result["elapsed"] = elapsed
        if logger:
            logger.info("%s completed in %.3f s", label, elapsed)


def to_uint8(image: np.ndarray) -> np.ndarray:
    """
    Safely rescale/clip a float or high-bit-depth image to uint8 [0, 255].

    Many stages internally work in float64 for numerical precision
    (especially the diffusion PDE). This helper is the single place that
    performs the final quantization back to a displayable/saveable image,
    avoiding inconsistent clipping logic scattered across modules.
    """
    img = image.astype(np.float64)
    img = np.clip(img, 0.0, 255.0 if img.max() > 1.5 else 1.0)
    if img.max() <= 1.0:
        img = img * 255.0
    return np.round(img).astype(np.uint8)


def normalize_to_unit_range(image: np.ndarray) -> np.ndarray:
    """Scale an arbitrary-range image to float64 in [0, 1]."""
    img = image.astype(np.float64)
    min_val, max_val = img.min(), img.max()
    if max_val - min_val < 1e-8:
        return np.zeros_like(img)
    return (img - min_val) / (max_val - min_val)


In [ ]:
%%writefile image_loader.py
"""
image_loader.py
================
Handles all input concerns: reading image files from disk (PNG, JPG,
JPEG, BMP, TIFF), validating that they are usable, and converting them
to grayscale. This is kept separate from preprocessing.py because
"loading + validation" is an I/O concern while "preprocessing" is a
numerical-transform concern -- separating them keeps each module
testable in isolation.
"""

from __future__ import annotations

from pathlib import Path
from typing import Union

import cv2
import numpy as np

from config import SUPPORTED_EXTENSIONS
from utils import get_logger

logger = get_logger(__name__)


class ImageLoadError(Exception):
    """Raised when an image cannot be read or fails validation."""


def validate_path(path: Union[str, Path]) -> Path:
    """
    Confirm the file exists and has a supported extension.

    Raising early with a clear message is much friendlier (especially in
    a GUI) than letting OpenCV silently return None deep in the pipeline.
    """
    p = Path(path)
    if not p.exists():
        raise ImageLoadError(f"File not found: {p}")
    if p.suffix.lower() not in SUPPORTED_EXTENSIONS:
        raise ImageLoadError(
            f"Unsupported file type '{p.suffix}'. "
            f"Supported types: {', '.join(SUPPORTED_EXTENSIONS)}"
        )
    return p


def load_image(path: Union[str, Path], as_grayscale: bool = True) -> np.ndarray:
    """
    Load an image from disk as a numpy array.

    Parameters
    ----------
    path : str or Path
        Path to a PNG/JPG/JPEG/BMP/TIFF file.
    as_grayscale : bool
        X-ray images are inherently single-channel. If the source file is
        stored as RGB (e.g. an exported/annotated JPEG), we convert it to
        grayscale using luminance weighting so downstream stages always
        receive a consistent 2D array.

    Returns
    -------
    np.ndarray
        uint8 array, shape (H, W) if as_grayscale else (H, W, 3).
    """
    p = validate_path(path)

    # IMREAD_UNCHANGED preserves bit depth (some clinical TIFFs are 16-bit);
    # we then normalize down to 8-bit for the rest of the pipeline, which
    # is written against uint8 for simplicity and matches typical exported
    # portable X-ray images.
    image = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)

    if image is None:
        raise ImageLoadError(
            f"OpenCV failed to decode '{p.name}'. The file may be corrupted "
            f"or in an unsupported encoding."
        )

    if image.dtype != np.uint8:
        image = _rescale_bit_depth(image)

    if as_grayscale and image.ndim == 3:
        # OpenCV loads as BGR; cvtColor handles the correct luminance weights.
        channels = image.shape[2]
        if channels == 4:
            image = cv2.cvtColor(image, cv2.COLOR_BGRA2GRAY)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    logger.info("Loaded image '%s' with shape %s, dtype %s", p.name, image.shape, image.dtype)
    return image


def _rescale_bit_depth(image: np.ndarray) -> np.ndarray:
    """
    Rescale non-uint8 images (commonly uint16 from DICOM-exported TIFFs)
    down to the 0-255 range using min-max normalization, preserving
    relative intensity relationships rather than naive truncation
    (which would clip most of the dynamic range).
    """
    img = image.astype(np.float64)
    min_val, max_val = img.min(), img.max()
    if max_val - min_val < 1e-8:
        return np.zeros(image.shape, dtype=np.uint8)
    img = (img - min_val) / (max_val - min_val) * 255.0
    return img.astype(np.uint8)


def validate_image_content(image: np.ndarray) -> None:
    """
    Sanity-check the decoded image before it enters the pipeline.

    Catches common failure modes: empty arrays, degenerate (all-black /
    all-white) images, and images too small to process meaningfully.
    """
    if image is None or image.size == 0:
        raise ImageLoadError("Decoded image is empty.")

    if image.shape[0] < 32 or image.shape[1] < 32:
        raise ImageLoadError(
            f"Image too small ({image.shape[0]}x{image.shape[1]}); "
            f"minimum 32x32 required for meaningful diffusion filtering."
        )

    if image.max() == image.min():
        raise ImageLoadError(
            "Image has zero dynamic range (constant pixel value); "
            "this is not a valid X-ray image."
        )

    logger.debug("Image content validated: shape=%s, range=[%d, %d]",
                 image.shape, int(image.min()), int(image.max()))


In [ ]:
%%writefile preprocessing.py
"""
preprocessing.py
=================
Numerical preprocessing that happens *before* the two denoising stages:

    normalize -> optional resize -> noise analysis

Why each step is needed
------------------------
* Normalization: the anisotropic diffusion PDE and the CNN denoiser both
  expect a known, bounded numeric range. Working in float64 [0, 1]
  avoids overflow/underflow in the gradient computations and matches
  the range most pretrained denoising CNNs (including DnCNN) were
  trained on.
* Resizing: portable/handheld X-ray units sometimes export very large
  images (>3000 px). Anisotropic diffusion is O(H*W*iterations), so an
  optional downscale keeps processing time reasonable on a laptop CPU
  without materially affecting diagnostic-relevant structures, which are
  generally much larger than a few pixels. Resizing is capped and
  optional -- disabled by setting resize_max_dim=0 in config.py.
* Noise analysis: estimating the noise level up front lets later stages
  (diffusion iteration count, DnCNN vs. skip) adapt instead of using a
  single hard-coded assumption. It's also directly useful diagnostic
  information to show the user in the GUI.
"""

from __future__ import annotations

from dataclasses import dataclass

import cv2
import numpy as np
from skimage.restoration import estimate_sigma

from config import PipelineConfig
from utils import get_logger, normalize_to_unit_range

logger = get_logger(__name__)


@dataclass
class NoiseAnalysisResult:
    estimated_sigma: float           # estimated Gaussian noise std-dev (0-255 scale)
    noise_level: str                 # qualitative bucket: low / moderate / high
    laplacian_variance: float        # a secondary sharpness/noise proxy


def normalize_image(image: np.ndarray) -> np.ndarray:
    """
    Convert a uint8 grayscale image to float64 in [0, 1].

    Working in floating point prevents the integer clipping/wraparound
    artifacts that would otherwise appear when the diffusion PDE computes
    small incremental updates on uint8 data.
    """
    return normalize_to_unit_range(image)


def resize_if_needed(image: np.ndarray, max_dim: int) -> tuple[np.ndarray, float]:
    """
    Downscale the image if its largest dimension exceeds `max_dim`.

    Returns the (possibly resized) image and the scale factor applied,
    so callers can later upscale results back to the original resolution
    if needed for reporting/saving at native size.
    """
    if max_dim <= 0:
        return image, 1.0

    h, w = image.shape[:2]
    longest = max(h, w)
    if longest <= max_dim:
        return image, 1.0

    scale = max_dim / float(longest)
    new_size = (int(round(w * scale)), int(round(h * scale)))
    resized = cv2.resize(image, new_size, interpolation=cv2.INTER_AREA)
    logger.info("Resized image from %s to %s (scale=%.3f)", (h, w), resized.shape[:2], scale)
    return resized, scale


def analyze_noise(image_uint8: np.ndarray) -> NoiseAnalysisResult:
    """
    Estimate the noise level present in the (grayscale, uint8) image.

    Two complementary estimators are used:
    1. skimage's wavelet-based robust sigma estimator (Donoho & Johnstone),
       which is standard for estimating additive Gaussian noise without
       needing a clean reference image.
    2. Variance of the Laplacian, a classic sharpness/blur-vs-noise proxy
       used as a sanity cross-check and for quick GUI feedback.

    This estimate is used to decide, e.g., how many diffusion iterations
    to recommend and whether the deep denoiser is likely to help.
    """
    img_float = image_uint8.astype(np.float64) / 255.0
    sigma = estimate_sigma(img_float, average_sigmas=True) * 255.0

    laplacian_var = cv2.Laplacian(image_uint8, cv2.CV_64F).var()

    if sigma < 5:
        level = "low"
    elif sigma < 15:
        level = "moderate"
    else:
        level = "high"

    result = NoiseAnalysisResult(
        estimated_sigma=float(sigma),
        noise_level=level,
        laplacian_variance=float(laplacian_var),
    )
    logger.info("Noise analysis: sigma=%.2f (%s), laplacian_var=%.2f",
                result.estimated_sigma, result.noise_level, result.laplacian_variance)
    return result


def preprocess(image_uint8: np.ndarray, config: PipelineConfig) -> dict:
    """
    Run the full preprocessing stage and return everything downstream
    stages need, bundled in a dict for a simple, explicit hand-off.
    """
    resized, scale = resize_if_needed(image_uint8, config.resize_max_dim)
    noise_info = analyze_noise(resized)
    normalized = normalize_image(resized)

    return {
        "resized_uint8": resized,
        "scale_factor": scale,
        "noise_info": noise_info,
        "normalized_float": normalized,
    }


In [ ]:
%%writefile anisotropic_diffusion.py
"""
anisotropic_diffusion.py
=========================
Perona-Malik Anisotropic Diffusion (Perona & Malik, 1990).

MATHEMATICAL MODEL
-------------------
Anisotropic diffusion smooths an image I(x, y, t) by solving the PDE:

    dI/dt = div( c(x, y, t) * grad(I) )

where `grad(I)` is the image gradient and `c(x, y, t)` is a conductance
(diffusivity) function that is LARGE in flat, homogeneous regions (so
diffusion/smoothing proceeds freely, removing noise) and SMALL near
strong edges (so diffusion is suppressed there, preserving anatomical
boundaries such as bone edges, soft-tissue interfaces, and implant
outlines in an X-ray).

This is what makes it "anisotropic" (direction/location-dependent)
as opposed to isotropic Gaussian blurring, which smooths edges and
flat regions equally and therefore washes out diagnostically important
structure.

DISCRETE UPDATE (finite differences, 4-neighbourhood)
-------------------------------------------------------
At each iteration, for every pixel, we compute directional gradients
to its 4 neighbours (North, South, East, West):

    grad_N = I[i-1, j] - I[i, j]
    grad_S = I[i+1, j] - I[i, j]
    grad_E = I[i, j+1] - I[i, j]
    grad_W = I[i, j-1] - I[i, j]

Each gradient is weighted by a conductance coefficient computed from
that same gradient, then the image is updated:

    I_new = I + lambda * (c_N*grad_N + c_S*grad_S + c_E*grad_E + c_W*grad_W)

CONDUCTANCE FUNCTIONS
-----------------------
Two classic choices, both controlled by the "edge threshold" kappa:

    Option 1 (exponential, favors high-contrast edges):
        c(grad) = exp( -(grad / kappa)^2 )

    Option 2 (quadratic / Lorentzian, favors wide regions,
              better for preserving wider low-contrast structures):
        c(grad) = 1 / (1 + (grad / kappa)^2)

In both cases, as |grad| -> 0 (flat region), c -> 1 (full smoothing).
As |grad| -> large (strong edge), c -> 0 (smoothing halted).

STABILITY / STOPPING CRITERIA
--------------------------------
* lambda must satisfy lambda <= 0.25 for the explicit finite-difference
  scheme to remain numerically stable on a 4-neighbour 2D stencil
  (standard CFL-type stability bound for this discretization).
* Iteration count is the practical stopping criterion: too few
  iterations leave residual noise, too many over-smooth fine
  anatomical detail and can introduce staircase artifacts. This
  implementation also supports an optional convergence check (mean
  absolute update below a small epsilon) so it can stop early.

TIME COMPLEXITY
------------------
O(H * W * num_iterations): every iteration is a single pass over all
pixels doing constant-time neighbour arithmetic (with NumPy vectorized
across the whole image per iteration, not a Python-level pixel loop).
"""

from __future__ import annotations

import numpy as np

from config import AnisotropicDiffusionConfig
from utils import get_logger

logger = get_logger(__name__)


def _conductance(gradient: np.ndarray, kappa: float, option: int) -> np.ndarray:
    """Compute the conductance (diffusivity) coefficient for a gradient array."""
    if option == 1:
        return np.exp(-(gradient / kappa) ** 2)
    elif option == 2:
        return 1.0 / (1.0 + (gradient / kappa) ** 2)
    else:
        raise ValueError(f"Unknown conductance option: {option} (expected 1 or 2)")


def anisotropic_diffusion(
    image: np.ndarray,
    config: AnisotropicDiffusionConfig,
    convergence_epsilon: float = 1e-4,
) -> np.ndarray:
    """
    Apply Perona-Malik anisotropic diffusion to a float image in [0, 1].

    Parameters
    ----------
    image : np.ndarray
        2D float array in [0, 1] (grayscale).
    config : AnisotropicDiffusionConfig
        num_iterations, kappa, lam, option — see config.py docstrings.
    convergence_epsilon : float
        If the mean absolute per-pixel update drops below this value,
        diffusion stops early (the image has effectively converged),
        which both saves time and avoids unnecessary over-smoothing.

    Returns
    -------
    np.ndarray
        Diffused float image in [0, 1], same shape as input.
    """
    if config.lam > 0.25:
        logger.warning(
            "lambda=%.3f exceeds the 0.25 stability bound for explicit 2D "
            "diffusion; reducing to 0.25 to avoid numerical instability.",
            config.lam,
        )
    lam = min(config.lam, 0.25)

    img = image.astype(np.float64).copy()

    for it in range(config.num_iterations):
        # Directional gradients via np.roll (periodic shift), equivalent to
        # Neumann (zero-flux) boundary handling once combined below since
        # we only ever use interior differences and roll wraps consistently
        # for all four directions.
        north = np.roll(img, 1, axis=0) - img
        south = np.roll(img, -1, axis=0) - img
        east = np.roll(img, -1, axis=1) - img
        west = np.roll(img, 1, axis=1) - img

        c_n = _conductance(north, config.kappa / 255.0 if config.kappa > 1 else config.kappa, config.option)
        c_s = _conductance(south, config.kappa / 255.0 if config.kappa > 1 else config.kappa, config.option)
        c_e = _conductance(east, config.kappa / 255.0 if config.kappa > 1 else config.kappa, config.option)
        c_w = _conductance(west, config.kappa / 255.0 if config.kappa > 1 else config.kappa, config.option)

        update = lam * (c_n * north + c_s * south + c_e * east + c_w * west)
        img = img + update

        mean_abs_update = np.mean(np.abs(update))
        if mean_abs_update < convergence_epsilon:
            logger.info(
                "Anisotropic diffusion converged early at iteration %d/%d "
                "(mean|update|=%.6f)", it + 1, config.num_iterations, mean_abs_update
            )
            break

    img = np.clip(img, 0.0, 1.0)
    logger.info(
        "Anisotropic diffusion complete: iterations=%d, kappa=%.2f, lambda=%.3f, option=%d",
        it + 1, config.kappa, lam, config.option
    )
    return img


In [ ]:
%%writefile deep_denoiser.py
"""
deep_denoiser.py
=================
Deep-learning denoising stage.

WHY DnCNN INSTEAD OF CBDNet
-----------------------------
CBDNet (Guo et al., 2019, "Toward Convolutional Blind Denoising of Real
Photographs") is designed around a *realistic noise model* pipeline: it
estimates a per-pixel noise level map via a small CNN, then feeds both
the noisy image and that estimated noise map into a U-Net-style
denoiser, and it is normally trained on paired data generated through a
signal-dependent noise synthesis pipeline (shot + read noise + in-camera
ISP effects) that models consumer camera sensors. Reproducing that
faithfully requires: (a) a noise-estimation sub-network trained with its
own loss, (b) an asymmetric loss that penalizes under-estimation of
noise more than over-estimation, and (c) the camera-realistic synthetic
noise generator itself -- none of which has published, load-ready
pretrained weights for grayscale medical X-ray images, and reproducing
it from scratch is a multi-week training project on its own, well beyond
a "hybrid classical + deep" enhancement tool.

DnCNN (Zhang et al., 2017, "Beyond a Gaussian Denoiser") is a much
simpler residual-learning CNN: a stack of Conv-BatchNorm-ReLU blocks
that learns to predict the *noise residual* (noisy image minus clean
image) rather than the clean image directly, which converges faster and
is a very well-documented, widely reproduced architecture with multiple
public pretrained checkpoints for Gaussian noise removal. Portable X-ray
sensor noise is reasonably well approximated as signal-independent
Gaussian/Poisson-Gaussian noise after the anisotropic diffusion stage
has already removed the bulk of the structured noise, which is exactly
the regime DnCNN targets. This makes DnCNN the pragmatic, reliable
choice for this hybrid pipeline while remaining architecturally
upgradeable to CBDNet later (this module's `denoise()` function is the
only integration point the rest of the pipeline depends on).

ARCHITECTURE
------------
Input (1 channel) -> Conv3x3+ReLU
                   -> [Conv3x3 + BatchNorm + ReLU] x (depth - 2)
                   -> Conv3x3 (no activation)         -> predicted noise
Output = Input - predicted_noise                       (residual learning)

Residual learning is used because at initialization a near-identity
mapping is trivial to learn (predict ~0 noise), which empirically speeds
up convergence versus predicting the clean image directly.

IF NO PRETRAINED WEIGHTS ARE AVAILABLE
-----------------------------------------
This module checks for a checkpoint at config.DNCNN_WEIGHTS_PATH. If it
is missing, the deep-learning stage is SKIPPED (not simulated, not
substituted silently) and the pipeline continues using only anisotropic
diffusion + CLAHE, with a clear log/warning and a flag returned to the
caller so the GUI can inform the user. See README.md "Getting DnCNN
pretrained weights" for how to obtain or train a checkpoint.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn

from config import DnCNNConfig
from utils import get_logger

logger = get_logger(__name__)


class DnCNN(nn.Module):
    """
    DnCNN architecture (Zhang et al., 2017) for grayscale image denoising.

    Parameters
    ----------
    depth : int
        Total number of convolutional layers (17 for DnCNN-S, the
        standard single-noise-level variant).
    num_channels : int
        Number of feature channels in the hidden layers (64 in the
        original paper).
    """

    def __init__(self, depth: int = 17, num_channels: int = 64):
        super().__init__()
        layers = [
            nn.Conv2d(1, num_channels, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),
        ]
        for _ in range(depth - 2):
            layers.append(nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1, bias=False))
            layers.append(nn.BatchNorm2d(num_channels))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Conv2d(num_channels, 1, kernel_size=3, padding=1, bias=False))
        self.body = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Predict the noise residual; caller subtracts it from the input."""
        noise = self.body(x)
        return x - noise


@dataclass
class DenoiseResult:
    output: np.ndarray
    applied: bool                 # False if the stage was skipped (no weights)
    reason: Optional[str] = None  # explains why it was skipped, if applicable
    device: Optional[str] = None


def _select_device(use_gpu_if_available: bool) -> torch.device:
    if use_gpu_if_available and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def load_model(config: DnCNNConfig) -> tuple[Optional[DnCNN], Optional[torch.device]]:
    """
    Instantiate the DnCNN architecture and load pretrained weights if the
    checkpoint file exists. Returns (None, None) if weights are absent so
    callers can skip the stage rather than run an untrained (random-weight)
    network, which would corrupt the image rather than denoise it.
    """
    weights_path = Path(config.weights_path)
    if not weights_path.exists():
        logger.warning(
            "DnCNN checkpoint not found at '%s'. Deep-learning denoising "
            "stage will be skipped. See README.md for how to obtain "
            "pretrained weights.", weights_path
        )
        return None, None

    device = _select_device(config.use_gpu_if_available)
    model = DnCNN(depth=config.depth, num_channels=config.num_channels)
    try:
        state_dict = torch.load(weights_path, map_location=device)
        # Support checkpoints saved either as a raw state_dict or wrapped
        # in a dict with a "state_dict" key (common training-script convention).
        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        model.load_state_dict(state_dict)
    except Exception as exc:  # noqa: BLE001 - we want to degrade gracefully on any load error
        logger.error("Failed to load DnCNN weights from '%s': %s", weights_path, exc)
        return None, None

    model.to(device)
    model.eval()
    logger.info("DnCNN loaded on device '%s' (depth=%d, channels=%d)",
                device, config.depth, config.num_channels)
    return model, device


def denoise(image_float: np.ndarray, config: DnCNNConfig) -> DenoiseResult:
    """
    Run DnCNN inference on a float image in [0, 1].

    Preprocessing: the image is converted to a (1, 1, H, W) float32 tensor,
    matching the (batch, channel, height, width) layout PyTorch conv
    layers expect, with values kept in [0, 1] (the range DnCNN was
    trained on for its Gaussian-denoising variants).

    Postprocessing: the output tensor is squeezed back to a 2D numpy
    array and clipped to [0, 1] to guard against minor overshoot from
    the residual subtraction at pixel extremes.
    """
    model, device = load_model(config)
    if model is None:
        return DenoiseResult(output=image_float, applied=False,
                              reason="No pretrained DnCNN checkpoint found")

    with torch.no_grad():
        tensor = torch.from_numpy(image_float.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(device)
        output_tensor = model(tensor)
        output = output_tensor.squeeze(0).squeeze(0).cpu().numpy()

    output = np.clip(output, 0.0, 1.0)
    return DenoiseResult(output=output, applied=True, device=str(device))


In [ ]:
%%writefile clahe.py
"""
clahe.py
========
Contrast Limited Adaptive Histogram Equalization (CLAHE).

ADAPTIVE HISTOGRAM EQUALIZATION (AHE)
----------------------------------------
Standard (global) histogram equalization computes one cumulative
distribution function (CDF) for the entire image and remaps every pixel
through it. For X-rays this is a problem: a portable chest/limb X-ray
typically contains both very dense regions (bone, metal implants) and
very faint regions (soft tissue) in the same frame, so a single global
mapping over-stretches contrast in one area while barely enhancing the
other.

AHE instead divides the image into small tiles, computes a separate
histogram/CDF per tile, and equalizes each tile independently, then
bilinearly interpolates between tile mappings at the boundaries to avoid
blocking artifacts. This lets faint soft-tissue detail become locally
more visible without blowing out already-bright bone regions.

CONTRAST LIMITING
--------------------
Plain AHE tends to over-amplify noise in near-flat tiles, because a
small amount of noise in an otherwise flat histogram gets stretched
across the full range. CLAHE fixes this by clipping the tile histogram
at a `clip_limit` before computing the CDF: any histogram bin count
above the limit is clipped, and the clipped mass is redistributed
uniformly across all bins. This bounds how aggressively any single tile
can be contrast-stretched, which is exactly why CLAHE (rather than plain
AHE) is the standard choice for medical image enhancement.

PARAMETER TUNING
-------------------
* clip_limit: higher = more contrast enhancement but more amplified
  noise. Typical medical-imaging range: 1.5-4.0. Values above ~5 tend to
  create visible tile-boundary artifacts and unrealistic contrast.
* tile_grid_size: smaller tiles (e.g. 4x4) enhance very local contrast
  but are more prone to noise amplification and blocking artifacts;
  larger tiles (e.g. 16x16) behave closer to global equalization. 8x8 is
  the OpenCV default and a reasonable starting point for typical X-ray
  resolutions.

Because the deep-denoising and diffusion stages run *before* CLAHE in
this pipeline, most of the noise that CLAHE would otherwise amplify has
already been suppressed, which is precisely why the pipeline order is
denoise-then-contrast-enhance rather than the reverse.
"""

from __future__ import annotations

import cv2
import numpy as np

from config import CLAHEConfig
from utils import get_logger

logger = get_logger(__name__)


def apply_clahe(image_uint8: np.ndarray, config: CLAHEConfig) -> np.ndarray:
    """
    Apply CLAHE to a uint8 grayscale image.

    Parameters
    ----------
    image_uint8 : np.ndarray
        2D uint8 grayscale image.
    config : CLAHEConfig
        clip_limit and tile_grid_size (see module docstring for tuning guidance).

    Returns
    -------
    np.ndarray
        uint8 grayscale image with locally enhanced contrast.
    """
    if image_uint8.dtype != np.uint8:
        raise ValueError("apply_clahe expects a uint8 image; convert with utils.to_uint8 first.")

    clahe = cv2.createCLAHE(
        clipLimit=config.clip_limit,
        tileGridSize=tuple(config.tile_grid_size),
    )
    enhanced = clahe.apply(image_uint8)
    logger.info("CLAHE applied: clip_limit=%.2f, tile_grid_size=%s",
                config.clip_limit, config.tile_grid_size)
    return enhanced


In [ ]:
%%writefile enhancement.py
"""
enhancement.py
===============
Optional edge-enhancement (sharpening) stage, applied after CLAHE.

METHODS
-------
1. Unsharp Mask: blurred = GaussianBlur(image); sharpened = image +
   amount * (image - blurred). This amplifies high-frequency detail
   (edges) that survives after subtracting a blurred version, without
   introducing the ringing that a naive Laplacian kernel can produce on
   already-noisy medical images.

2. Laplacian Sharpening: sharpened = image - laplacian_gain *
   Laplacian(image). The Laplacian is a second-derivative operator, so
   it responds strongly to fine edges/noise alike, making it more
   aggressive and more noise-sensitive than unsharp masking.

WHEN SHARPENING SHOULD BE AVOIDED
-------------------------------------
Sharpening amplifies high-frequency content, and noise IS high-frequency
content. If the denoising stages upstream left any residual noise (or if
the source image is already very high-resolution/high-detail),
sharpening will amplify that noise right alongside real anatomical
edges, which can create false structures or exaggerate artifacts -- a
real risk in a diagnostic-support tool. For that reason this stage is:
  (a) OFF by default unless explicitly enabled,
  (b) auto-guarded: after sharpening, the local noise proxy is
      recomputed, and if it increased beyond `edge_gain_guard`
      (a configurable multiplier) relative to the pre-sharpen image,
      the sharpened result is discarded and the CLAHE output is passed
      through unchanged. This is a simple, deterministic policy
      appropriate for a decision-support tool that should never
      silently trade real anatomical clarity for noise amplification.
"""

from __future__ import annotations

from dataclasses import dataclass

import cv2
import numpy as np

from config import SharpeningConfig
from utils import get_logger

logger = get_logger(__name__)


@dataclass
class SharpenResult:
    output: np.ndarray
    applied: bool
    reason: str = ""


def _unsharp_mask(image_uint8: np.ndarray, amount: float, radius: float) -> np.ndarray:
    blurred = cv2.GaussianBlur(image_uint8, ksize=(0, 0), sigmaX=radius)
    sharpened = cv2.addWeighted(image_uint8, 1 + amount, blurred, -amount, 0)
    return sharpened


def _laplacian_sharpen(image_uint8: np.ndarray, amount: float) -> np.ndarray:
    laplacian = cv2.Laplacian(image_uint8, cv2.CV_64F, ksize=3)
    sharpened = image_uint8.astype(np.float64) - amount * laplacian
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def _noise_proxy(image_uint8: np.ndarray) -> float:
    """
    Fast local-noise proxy used only to decide whether sharpening should
    be kept or discarded (NOT the same as the full evaluation-stage
    metrics in evaluation.py, which require an original reference).
    Uses the high-frequency residual energy after a small median blur.
    """
    denoised_ref = cv2.medianBlur(image_uint8, 3)
    residual = image_uint8.astype(np.float64) - denoised_ref.astype(np.float64)
    return float(np.std(residual))


def apply_sharpening(image_uint8: np.ndarray, config: SharpeningConfig) -> SharpenResult:
    """
    Apply the configured sharpening method, guarded against noise
    amplification. See module docstring for the guard rationale.
    """
    if not config.enabled:
        return SharpenResult(output=image_uint8, applied=False, reason="Sharpening disabled by config")

    pre_noise = _noise_proxy(image_uint8)

    if config.method == "unsharp_mask":
        sharpened = _unsharp_mask(image_uint8, config.amount, config.radius)
    elif config.method == "laplacian":
        sharpened = _laplacian_sharpen(image_uint8, config.amount)
    else:
        raise ValueError(f"Unknown sharpening method: {config.method}")

    post_noise = _noise_proxy(sharpened)

    if pre_noise > 1e-6 and (post_noise / pre_noise) > config.edge_gain_guard:
        logger.warning(
            "Sharpening increased noise proxy by %.2fx (limit %.2fx); "
            "discarding sharpened result and keeping the pre-sharpen image.",
            post_noise / pre_noise, config.edge_gain_guard
        )
        return SharpenResult(
            output=image_uint8, applied=False,
            reason=f"Auto-skipped: noise increased {post_noise / pre_noise:.2f}x"
        )

    logger.info("Sharpening applied (%s): noise proxy %.2f -> %.2f",
                config.method, pre_noise, post_noise)
    return SharpenResult(output=sharpened, applied=True)


In [ ]:
%%writefile evaluation.py
"""
evaluation.py
==============
Image quality evaluation metrics computed after the pipeline finishes.

METRICS
-------
* MSE  (Mean Squared Error): average squared per-pixel difference
  between original and enhanced images. Lower = more similar to the
  input. Simple, but does not correlate well with perceived quality on
  its own -- included mainly as the basis for PSNR/RMSE and as a
  familiar baseline number.

* RMSE (Root Mean Squared Error): sqrt(MSE), same units as pixel
  intensity (0-255), which makes it more directly interpretable than
  MSE ("images differ by ~X intensity levels on average").

* PSNR (Peak Signal-to-Noise Ratio): 10*log10(MAX^2 / MSE), expressed in
  dB. Widely used because it is a standard, easily comparable number
  across denoising literature. Caveat: PSNR is computed against the
  *original noisy* image here (not a clean ground truth, which doesn't
  exist for real X-rays), so it should be read as "how much the pixel
  values changed", not "how much noise was removed" in an absolute
  sense -- SSIM and entropy below help interpret it correctly.

* SSIM (Structural Similarity Index): compares local luminance,
  contrast, and structure between two images, in the range [-1, 1] (1 =
  identical). Unlike MSE/PSNR, SSIM is designed to track *perceived*
  structural similarity, which is more clinically relevant: a good
  enhancement should keep SSIM reasonably high (structure/edges
  preserved) even though pixel values changed a lot (noise removed,
  contrast boosted).

* Entropy: Shannon entropy of the intensity histogram, in bits. Higher
  entropy generally indicates more information/detail content. Used
  here to check that denoising didn't oversmooth away real detail
  (entropy dropping sharply is a warning sign) while contrast
  enhancement often *raises* entropy by spreading the histogram.

* Edge Preservation Index (EPI, optional): correlation between the
  Laplacian-filtered original and Laplacian-filtered enhanced image,
  restricted to edge regions. Values near 1 indicate edges were well
  preserved; values near 0 indicate edges were blurred away. This is
  the single metric most directly tied to "did we keep the anatomical
  boundaries intact", which is the core goal for a diagnostic tool.

All metrics are computed on uint8 [0, 255] images.
"""

from __future__ import annotations

from dataclasses import dataclass, asdict

import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric

from utils import get_logger

logger = get_logger(__name__)


@dataclass
class QualityMetrics:
    mse: float
    rmse: float
    psnr: float
    ssim: float
    entropy_original: float
    entropy_enhanced: float
    edge_preservation_index: float

    def as_dict(self) -> dict:
        return asdict(self)


def _shannon_entropy(image_uint8: np.ndarray) -> float:
    hist = cv2.calcHist([image_uint8], [0], None, [256], [0, 256]).flatten()
    hist = hist / (hist.sum() + 1e-12)
    hist = hist[hist > 0]
    return float(-np.sum(hist * np.log2(hist)))


def _edge_preservation_index(original: np.ndarray, enhanced: np.ndarray) -> float:
    """
    Correlation coefficient between Laplacian edge maps of the original
    and enhanced images. Implemented directly (rather than relying on a
    single library call) so the definition used here is explicit and
    auditable, which matters for a metric used to judge diagnostic
    fidelity.
    """
    lap_orig = cv2.Laplacian(original, cv2.CV_64F, ksize=3)
    lap_enh = cv2.Laplacian(enhanced, cv2.CV_64F, ksize=3)

    lap_orig_flat = lap_orig.flatten() - lap_orig.mean()
    lap_enh_flat = lap_enh.flatten() - lap_enh.mean()

    numerator = np.sum(lap_orig_flat * lap_enh_flat)
    denominator = np.sqrt(np.sum(lap_orig_flat ** 2) * np.sum(lap_enh_flat ** 2)) + 1e-12
    return float(numerator / denominator)


def evaluate(original_uint8: np.ndarray, enhanced_uint8: np.ndarray) -> QualityMetrics:
    """
    Compute the full quality-metric suite comparing the enhanced image
    against the original input image (resized to match if needed).
    """
    if original_uint8.shape != enhanced_uint8.shape:
        original_uint8 = cv2.resize(
            original_uint8, (enhanced_uint8.shape[1], enhanced_uint8.shape[0]),
            interpolation=cv2.INTER_AREA
        )

    orig_f = original_uint8.astype(np.float64)
    enh_f = enhanced_uint8.astype(np.float64)

    mse = float(np.mean((orig_f - enh_f) ** 2))
    rmse = float(np.sqrt(mse))

    if mse < 1e-10:
        psnr = float("inf")
    else:
        psnr = float(psnr_metric(original_uint8, enhanced_uint8, data_range=255))

    ssim_value = float(ssim_metric(original_uint8, enhanced_uint8, data_range=255))

    entropy_original = _shannon_entropy(original_uint8)
    entropy_enhanced = _shannon_entropy(enhanced_uint8)

    epi = _edge_preservation_index(original_uint8, enhanced_uint8)

    metrics = QualityMetrics(
        mse=mse,
        rmse=rmse,
        psnr=psnr,
        ssim=ssim_value,
        entropy_original=entropy_original,
        entropy_enhanced=entropy_enhanced,
        edge_preservation_index=epi,
    )

    logger.info(
        "Quality metrics: MSE=%.3f RMSE=%.3f PSNR=%.2fdB SSIM=%.4f "
        "Entropy(orig/enh)=%.3f/%.3f EPI=%.4f",
        metrics.mse, metrics.rmse, metrics.psnr, metrics.ssim,
        metrics.entropy_original, metrics.entropy_enhanced, metrics.edge_preservation_index,
    )
    return metrics


In [ ]:
%%writefile pipeline.py
"""
pipeline.py
============
Orchestrates the complete hybrid enhancement pipeline, stage by stage,
so both the GUI (gui.py) and CLI (main.py) share one authoritative code
path rather than duplicating the sequencing logic.

PIPELINE ORDER AND WHY EACH STAGE IS NECESSARY
--------------------------------------------------
1. Image Validation      - fail fast on corrupt/unusable input before
                            any processing time is spent.
2. Grayscale Conversion   - X-rays are single-channel; guarantees every
                            downstream stage sees a consistent 2D array.
3. Noise Analysis         - estimates noise level to inform iteration
                            counts and to report to the user; does not
                            modify the image.
4. Anisotropic Diffusion  - a classical, edge-aware first pass that
                            removes a large fraction of noise while
                            explicitly preserving strong anatomical
                            edges, using a fast, interpretable PDE model
                            with no training data required.
5. DnCNN Deep Denoiser    - a learned residual denoiser that captures
                            complex noise statistics classical filters
                            cannot model, cleaning up what diffusion
                            alone leaves behind (subtle/textured noise).
                            Skipped gracefully if no pretrained weights
                            are available (see deep_denoiser.py).
6. CLAHE                  - after noise is suppressed, locally boosts
                            contrast so faint structures become visible,
                            without over-amplifying noise (which running
                            CLAHE *before* denoising would risk).
7. Optional Sharpening    - a final, guarded pass that can sharpen edges
                            further if it does not reintroduce noise;
                            auto-skipped otherwise.
8. Quality Evaluation     - quantifies what stages 4-7 actually did
                            (PSNR/SSIM/MSE/RMSE/Entropy/EPI) so the
                            result is not just visually judged.

Each stage is implemented in its own module; this file only sequences
them and collects timing/metadata for the GUI and reports.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np

import anisotropic_diffusion
import clahe as clahe_module
import deep_denoiser
import enhancement
import evaluation
import preprocessing
from config import PipelineConfig, DEFAULT_CONFIG
from image_loader import load_image, validate_image_content
from utils import get_logger, timer, to_uint8

logger = get_logger(__name__)


@dataclass
class StageTiming:
    name: str
    seconds: float


@dataclass
class PipelineResult:
    original_uint8: np.ndarray            # input, after grayscale conversion + resize
    diffused_uint8: np.ndarray             # after anisotropic diffusion
    denoised_uint8: np.ndarray             # after DnCNN (or same as diffused if skipped)
    clahe_uint8: np.ndarray                # after CLAHE
    final_uint8: np.ndarray                # after optional sharpening (the deliverable)
    difference_uint8: np.ndarray           # |original - final|, contrast-stretched for display
    noise_info: preprocessing.NoiseAnalysisResult
    dncnn_applied: bool
    dncnn_skip_reason: Optional[str]
    sharpen_applied: bool
    sharpen_skip_reason: Optional[str]
    metrics: evaluation.QualityMetrics
    stage_timings: list = field(default_factory=list)
    total_seconds: float = 0.0


def _difference_image(original: np.ndarray, final: np.ndarray) -> np.ndarray:
    """Absolute difference image, contrast-stretched to [0, 255] for visibility."""
    diff = np.abs(original.astype(np.float64) - final.astype(np.float64))
    if diff.max() > 1e-6:
        diff = diff / diff.max() * 255.0
    return diff.astype(np.uint8)


def run_pipeline(
    image_path: str | Path,
    config: PipelineConfig = DEFAULT_CONFIG,
) -> PipelineResult:
    """
    Execute the full pipeline on an image file path and return every
    intermediate result needed for display, saving, and reporting.
    """
    stage_timings: list[StageTiming] = []
    total_start_seconds = 0.0

    with timer("Total pipeline", logger) as total_t:
        # --- Stage 1-2: Load, validate, grayscale -------------------------
        with timer("Image loading + validation", logger) as t:
            raw = load_image(image_path, as_grayscale=True)
            validate_image_content(raw)
        stage_timings.append(StageTiming("Load & Validate", t["elapsed"]))

        # --- Stage 3: Preprocessing (resize, normalize) + noise analysis -
        with timer("Preprocessing + noise analysis", logger) as t:
            prep = preprocessing.preprocess(raw, config)
        stage_timings.append(StageTiming("Preprocess & Noise Analysis", t["elapsed"]))

        original_uint8 = prep["resized_uint8"]
        normalized = prep["normalized_float"]

        # --- Stage 4: Anisotropic diffusion --------------------------------
        with timer("Anisotropic diffusion", logger) as t:
            diffused_float = anisotropic_diffusion.anisotropic_diffusion(normalized, config.anisotropic)
            diffused_uint8 = to_uint8(diffused_float)
        stage_timings.append(StageTiming("Anisotropic Diffusion", t["elapsed"]))

        # --- Stage 5: Deep learning denoiser (DnCNN) -----------------------
        with timer("DnCNN denoising", logger) as t:
            dncnn_result = deep_denoiser.denoise(diffused_float, config.dncnn)
            denoised_uint8 = to_uint8(dncnn_result.output)
        stage_timings.append(StageTiming("DnCNN Denoiser", t["elapsed"]))

        # --- Stage 6: CLAHE --------------------------------------------------
        with timer("CLAHE", logger) as t:
            clahe_uint8 = clahe_module.apply_clahe(denoised_uint8, config.clahe)
        stage_timings.append(StageTiming("CLAHE", t["elapsed"]))

        # --- Stage 7: Optional sharpening ------------------------------------
        with timer("Optional sharpening", logger) as t:
            sharpen_result = enhancement.apply_sharpening(clahe_uint8, config.sharpen)
            final_uint8 = sharpen_result.output
        stage_timings.append(StageTiming("Optional Sharpening", t["elapsed"]))

        # --- Stage 8: Quality evaluation --------------------------------------
        with timer("Quality evaluation", logger) as t:
            metrics = evaluation.evaluate(original_uint8, final_uint8)
        stage_timings.append(StageTiming("Quality Evaluation", t["elapsed"]))

        difference_uint8 = _difference_image(original_uint8, final_uint8)

    result = PipelineResult(
        original_uint8=original_uint8,
        diffused_uint8=diffused_uint8,
        denoised_uint8=denoised_uint8,
        clahe_uint8=clahe_uint8,
        final_uint8=final_uint8,
        difference_uint8=difference_uint8,
        noise_info=prep["noise_info"],
        dncnn_applied=dncnn_result.applied,
        dncnn_skip_reason=dncnn_result.reason,
        sharpen_applied=sharpen_result.applied,
        sharpen_skip_reason=sharpen_result.reason,
        metrics=metrics,
        stage_timings=stage_timings,
        total_seconds=total_t["elapsed"],
    )
    logger.info("Pipeline finished in %.3f s", result.total_seconds)
    return result


## 3. Import the pipeline


In [ ]:
import importlib
import config, utils, image_loader, preprocessing
import anisotropic_diffusion, deep_denoiser, clahe, enhancement, evaluation, pipeline

for m in (config, utils, image_loader, preprocessing, anisotropic_diffusion,
          deep_denoiser, clahe, enhancement, evaluation, pipeline):
    importlib.reload(m)

print("Pipeline modules loaded successfully.")
print("PyTorch device available:", "cuda" if __import__('torch').cuda.is_available() else "cpu")


## 4. (Optional) Get pretrained DnCNN weights

The deep-learning stage looks for a checkpoint at `models/dncnn.pth`. If it isn't present, the pipeline **automatically skips** that stage (anisotropic diffusion + CLAHE still run) and tells you so — it never runs an untrained network on your image.

Options to obtain weights:
- Train your own using `train_dncnn.py` (Section 8 below) on a noisy/clean image pair dataset.
- Upload a compatible DnCNN state_dict (`1->64->...->1` conv stack, grayscale, residual learning) as `models/dncnn.pth` using the file browser on the left.


## 5. Upload an X-ray image


In [ ]:
from google.colab import files
import shutil

print("Choose an X-ray image file (PNG/JPG/JPEG/BMP/TIFF)...")
uploaded = files.upload()
assert len(uploaded) > 0, "No file uploaded."

uploaded_filename = list(uploaded.keys())[0]
INPUT_IMAGE_PATH = f"{PROJECT_DIR}/sample_images/{uploaded_filename}"
shutil.move(uploaded_filename, INPUT_IMAGE_PATH)
print("Saved to:", INPUT_IMAGE_PATH)


### No image to upload? Generate a synthetic test X-ray instead
Run this cell **instead of** the upload cell above if you just want to test the pipeline.


In [ ]:
import numpy as np
import cv2

np.random.seed(42)
h, w = 512, 512
yy, xx = np.mgrid[0:h, 0:w]
img = 90 + 20 * (yy / h)
for cy in range(60, 460, 40):
    arc = np.exp(-(((xx - w/2)**2)/(2*180**2) + ((yy - cy)**2)/(2*10**2)))
    img += 40 * arc
for cx in (w*0.33, w*0.67):
    ellipse = ((xx-cx)**2)/(120**2) + ((yy-h*0.5)**2)/(180**2) < 1
    img[ellipse] -= 35
img[np.abs(xx - w/2) < 15] += 60
img = np.clip(img, 0, 255)

scaled = img / 255.0 * 40
poisson_noisy = np.random.poisson(scaled) / 40 * 255.0
gaussian_noise = np.random.normal(0, 12, img.shape)
noisy = np.clip(poisson_noisy + gaussian_noise, 0, 255).astype(np.uint8)

INPUT_IMAGE_PATH = f"{PROJECT_DIR}/sample_images/synthetic_xray_noisy.png"
cv2.imwrite(INPUT_IMAGE_PATH, noisy)
print("Synthetic noisy X-ray phantom saved to:", INPUT_IMAGE_PATH)


## 6. Configure parameters (optional)
Adjust and re-run this cell before processing to change diffusion / CLAHE / sharpening behaviour.


In [ ]:
from config import PipelineConfig, AnisotropicDiffusionConfig, DnCNNConfig, CLAHEConfig, SharpeningConfig

pipeline_config = PipelineConfig(
    resize_max_dim=1024,
    anisotropic=AnisotropicDiffusionConfig(num_iterations=15, kappa=30.0, lam=0.20, option=1),
    dncnn=DnCNNConfig(depth=17, num_channels=64, use_gpu_if_available=True),
    clahe=CLAHEConfig(clip_limit=2.5, tile_grid_size=(8, 8)),
    sharpen=SharpeningConfig(enabled=True, method="unsharp_mask", amount=0.6, radius=2.0, edge_gain_guard=1.15),
)
print(pipeline_config)


## 7. Run the pipeline


In [ ]:
result = pipeline.run_pipeline(INPUT_IMAGE_PATH, pipeline_config)

print(f"Noise level detected: {result.noise_info.noise_level} (sigma={result.noise_info.estimated_sigma:.2f})")
print(f"DnCNN deep-denoising applied: {result.dncnn_applied}"
      + ("" if result.dncnn_applied else f"  (reason: {result.dncnn_skip_reason})"))
print(f"Sharpening applied: {result.sharpen_applied}"
      + ("" if result.sharpen_applied else f"  (reason: {result.sharpen_skip_reason})"))
print(f"Total processing time: {result.total_seconds:.2f} s")
print()
for st in result.stage_timings:
    print(f"  {st.name:30s} {st.seconds:.3f} s")


## 8. Display results


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles_images = [
    ("Original (noisy) input", result.original_uint8),
    ("After Anisotropic Diffusion", result.diffused_uint8),
    ("After DnCNN Denoiser", result.denoised_uint8),
    ("After CLAHE", result.clahe_uint8),
    ("Final Enhanced Output", result.final_uint8),
    ("Difference (Original vs Final)", result.difference_uint8),
]
for ax, (title, img) in zip(axes.flat, titles_images):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Histogram comparison: original vs enhanced
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(result.original_uint8.ravel(), bins=256, range=(0, 255), alpha=0.5, label="Original", color="steelblue")
ax.hist(result.final_uint8.ravel(), bins=256, range=(0, 255), alpha=0.5, label="Enhanced", color="darkorange")
ax.set_xlabel("Pixel intensity")
ax.set_ylabel("Frequency")
ax.set_title("Histogram Comparison")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Quality evaluation metrics


In [ ]:
m = result.metrics
print(f"{'Metric':<28}{'Value':>12}")
print("-" * 40)
print(f"{'MSE':<28}{m.mse:>12.3f}")
print(f"{'RMSE':<28}{m.rmse:>12.3f}")
print(f"{'PSNR (dB)':<28}{m.psnr:>12.3f}")
print(f"{'SSIM':<28}{m.ssim:>12.4f}")
print(f"{'Entropy (original, bits)':<28}{m.entropy_original:>12.3f}")
print(f"{'Entropy (enhanced, bits)':<28}{m.entropy_enhanced:>12.3f}")
print(f"{'Edge Preservation Index':<28}{m.edge_preservation_index:>12.4f}")


## 10. Save and download the enhanced image


In [ ]:
import cv2
from google.colab import files as colab_files

output_path = f"{PROJECT_DIR}/outputs/enhanced_output.png"
cv2.imwrite(output_path, result.final_uint8)
print("Saved:", output_path)

colab_files.download(output_path)


## 11. (Optional) Train a DnCNN checkpoint in Colab

This is a minimal training loop skeleton (synthetic Gaussian noise on your own clean images) you can extend with a real dataset. Requires clean reference images in `sample_images/clean/`.


In [ ]:
%%writefile train_dncnn.py
"""
train_dncnn.py
===============
Minimal training loop to produce a models/dncnn.pth checkpoint compatible
with deep_denoiser.DnCNN. Trains on synthetic Gaussian noise added to a
folder of clean grayscale images, which is the standard way DnCNN-style
denoisers are trained when no paired noisy/clean dataset exists.

Usage:
    python train_dncnn.py --clean_dir sample_images/clean --epochs 20
"""
import argparse
import glob
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from deep_denoiser import DnCNN
from config import DnCNNConfig, MODELS_DIR


def load_clean_patches(clean_dir: str, patch_size: int = 64, stride: int = 32):
    patches = []
    for path in glob.glob(str(Path(clean_dir) / "*")):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        h, w = img.shape
        for y in range(0, h - patch_size, stride):
            for x in range(0, w - patch_size, stride):
                patches.append(img[y:y + patch_size, x:x + patch_size])
    return np.stack(patches).astype(np.float32) / 255.0 if patches else np.empty((0, patch_size, patch_size), np.float32)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--clean_dir", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--noise_sigma", type=float, default=15.0, help="0-255 scale")
    parser.add_argument("--lr", type=float, default=1e-3)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    patches = load_clean_patches(args.clean_dir)
    if len(patches) == 0:
        raise SystemExit(f"No usable images found in {args.clean_dir}")

    model = DnCNN(depth=17, num_channels=64).to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.MSELoss()

    n = len(patches)
    for epoch in range(args.epochs):
        perm = np.random.permutation(n)
        epoch_loss = 0.0
        for i in range(0, n, args.batch_size):
            idx = perm[i:i + args.batch_size]
            clean_batch = patches[idx]
            noise = np.random.normal(0, args.noise_sigma / 255.0, clean_batch.shape).astype(np.float32)
            noisy_batch = np.clip(clean_batch + noise, 0, 1)

            clean_t = torch.from_numpy(clean_batch).unsqueeze(1).to(device)
            noisy_t = torch.from_numpy(noisy_batch).unsqueeze(1).to(device)

            optimizer.zero_grad()
            output = model(noisy_t)
            loss = criterion(output, clean_t)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(idx)

        print(f"Epoch {epoch+1}/{args.epochs}  loss={epoch_loss/n:.6f}")

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    save_path = MODELS_DIR / "dncnn.pth"
    torch.save(model.state_dict(), save_path)
    print("Saved checkpoint to", save_path)


if __name__ == "__main__":
    main()


In [ ]:
# Example (uncomment and provide your own clean image folder to actually train):
# !python train_dncnn.py --clean_dir sample_images/clean --epochs 20
